# 04 — Audio Preprocessing

Prepare consistent audio features (64-band log-mel spectrograms) with explicit sample-rate and duration policies.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

In [ ]:
from pathlib import Path
import numpy as np

SAMPLE_RATE = 16_000
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256
CACHE_DIR = Path("../../data/processed/audio_mels")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Audio cache:", CACHE_DIR.resolve())

## Audio pipeline
Load the audio stream or associated audio file, convert to mono, resample to 16 kHz, compute a 64-band mel spectrogram, then apply a numerically stable log transform. Decide and document how variable-length clips are padded or cropped. Keep the policy identical across train/dev/test.

In [ ]:
try:
    import librosa
    print("librosa:", librosa.__version__)
except ImportError:
    print("Install librosa and a compatible audio backend before feature extraction.")

In [ ]:
def log_mel_from_waveform(waveform, sample_rate=SAMPLE_RATE):
    """Compute a 64-bin log-mel feature matrix from a mono waveform."""
    import librosa
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim != 1:
        raise ValueError("Expected a mono, one-dimensional waveform")
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

print("Feature function defined; run it only after loading a verified sample.")

### Validation checklist
Check waveform duration, sample rate, silence, clipping, feature dimensions, and a few spectrogram plots. Cache outputs by stable record ID and preserve a manifest for resuming after runtime restarts.